# Track A: Telco Churn — MLOps Pipeline

This notebook walks through the full MLOps pipeline:
1. Data preparation
2. Model training with MLflow tracking
3. Model registry & alias transitions
4. Evidently drift monitoring
5. Model serving

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

## 1. Data Preparation

In [ ]:
from src.track_a.data_prep import prepare_pipeline

data = prepare_pipeline()
print(f"Train: {data['X_train'].shape}, Test: {data['X_test'].shape}")
print(f"Churn rate: {data['y_train'].mean():.3f}")
data['X_train'].head()

## 2. Start MLflow Tracking Server

Run in a separate terminal:
```bash
cd track-a-churn
mkdir -p data
uv run mlflow server --backend-store-uri sqlite:///data/mlflow.db \
  --default-artifact-root ./data/mlruns --host 127.0.0.1 --port 5000
```

## 3. Train Models with MLflow

In [ ]:
from src.track_a.train import main as train_main
train_main()

## 4. Evidently Drift Monitoring

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from src.track_a.data_prep import load_raw, preprocess
from src.track_a.utils.evidently_reporter import EvidentlyReporter

df = load_raw()
df = preprocess(df)

reference_df, current_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['Churn'])

# Inject drift
current_df = current_df.copy()
rng = np.random.default_rng(42)
current_df['MonthlyCharges'] += rng.normal(15, 5, size=len(current_df))
current_df['tenure'] = (current_df['tenure'] + rng.normal(-10, 3, size=len(current_df))).clip(lower=0)
mtm_mask = current_df['Contract'] == 'Month-to-month'
current_df.loc[current_df[~mtm_mask].sample(frac=0.4, random_state=1).index, 'Contract'] = 'Month-to-month'

# Load best model for predictions
from src.track_a.utils.mlflow_utils import MLFlowLogger
logger = MLFlowLogger()
best_model = logger.load_model('ChurnClassifier', alias='production')

feature_cols = [c for c in df.columns if c != 'Churn']
current_df['prediction'] = best_model.predict(current_df[feature_cols])
reference_df['prediction'] = best_model.predict(reference_df[feature_cols])

reporter = EvidentlyReporter()
drift_results = reporter.generate_drift_report(reference_df, current_df)
print(f"Drift report saved: {drift_results['drift_path']}")

## 5. Run Drift Tests

In [ ]:
test_results = reporter.run_drift_tests(reference_df, current_df)
suite = test_results['test_suite']
print(f"Tests passed: {suite.passed_count()}/{suite.count()}")

## 6. Model Registry

Check the registered model and alias transitions:

In [ ]:
from mlflow import MlflowClient
client = MlflowClient(tracking_uri='http://127.0.0.1:5000')

# Check aliases
try:
    staging = client.get_model_version_by_alias('ChurnClassifier', 'staging')
    print(f"Staging: version {staging.version}")
except Exception as e:
    print(f"No staging alias: {e}")

try:
    prod = client.get_model_version_by_alias('ChurnClassifier', 'production')
    print(f"Production: version {prod.version}")
except Exception as e:
    print(f"No production alias: {e}")